# Medical PDF: Bronze to Silver

Attach the target schema-enabled Lakehouse as default. Set the Lakehouse ID and input path in the first code cell. Run all cells in order. Start with the generated synthetic fixture.

Run this PySpark notebook with `medical_documents` attached as the default Lakehouse. Files and tables use separate namespaces in Fabric:

- **Bronze:** Leave the source PDF unchanged in `Files/raw/`. `bronze.pdf_documents` is the Delta manifest, keyed by SHA-256 of the PDF bytes.
- **Silver:** `Files/silver/pdf_pages/<document_id>.json` contains the text and extraction status of each page. `silver.pdf_pages` is the queryable Delta equivalent, keyed by `(document_id, page_number)`. Page numbers start at 1.
- **Gold:** Not populated here. Clinical entities, normalized dates, and application-facing tables require a separate, validated transformation.

Only the embedded PDF text layer is read. No OCR, external AI service, or heuristic reversal of Hebrew text is used. `no_usable_text` marks blank or image-only pages for review; `review_text` marks text with replacement/private-use characters. These signals do not verify visual word order. The notebook prints aggregate QA only. Limit access to both the Silver JSON and the table: both can contain medical information.

In [ ]:
from pathlib import PurePosixPath
from uuid import UUID
import notebookutils

LAKEHOUSE_ID = '<target-lakehouse-guid>'
SOURCE_PATH = 'Files/raw/synthetic-medical-record.pdf'

try:
    expected_lakehouse = str(UUID(LAKEHOUSE_ID))
except (ValueError, TypeError) as error:
    raise ValueError('Set LAKEHOUSE_ID to the target default Lakehouse item GUID.') from error
if str(notebookutils.runtime.context['defaultLakehouseId']).lower() != expected_lakehouse:
    raise RuntimeError('Attach the configured Lakehouse as default and restart the session.')
path = PurePosixPath(SOURCE_PATH)
if path.parts[:2] != ('Files', 'raw') or '..' in path.parts or chr(92) in SOURCE_PATH or path.suffix.lower() != '.pdf':
    raise ValueError('SOURCE_PATH must be a relative PDF path under Files/raw/.')
SOURCE_PATH = path.as_posix()


In [ ]:
%pip install --quiet pypdf==7.1.1

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import json
import logging
from unicodedata import category

import pypdf
from pypdf.errors import PdfReadError

LAKEHOUSE = Path('/lakehouse/default')
MAX_BYTES = 50 * 1024 * 1024
EXTRACTOR = f'pypdf/{pypdf.__version__}'

def extract_pdf(data):
    logging.getLogger('pypdf').setLevel(logging.CRITICAL)
    try:
        reader = pypdf.PdfReader(BytesIO(data))
        if reader.is_encrypted:
            raise ValueError('Encrypted PDF: supply an authorized unencrypted document.')
        pages = []
        for number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ''
            characters = sum(char.isalnum() for char in text)
            hebrew = sum('\u05d0' <= char <= '\u05ea' for char in text)
            suspect = sum(char == '\ufffd' or category(char) == 'Co' for char in text)
            status = 'no_usable_text' if not characters else 'review_text' if suspect else 'text_extracted'
            pages.append(dict(page_number=number, text=text, extraction_status=status,
                              alphanumeric_count=characters, hebrew_letter_count=hebrew,
                              suspect_character_count=suspect))
    except PdfReadError:
        raise ValueError('PDF parsing failed; no Silver data was written by this run.') from None
    if not pages:
        raise ValueError('The PDF has no pages.')
    return pages

with (LAKEHOUSE / SOURCE_PATH).open('rb') as source:
    data = source.read(MAX_BYTES + 1)
if not data or len(data) > MAX_BYTES:
    raise ValueError('Expected a nonempty PDF no larger than 50 MiB.')
doc_id = sha256(data).hexdigest()
byte_size = len(data)
pages = extract_pdf(data)
del data
page_counts = Counter(page['extraction_status'] for page in pages)
print({'pages': len(pages), 'text_status_counts': dict(page_counts),
       'pages_with_hebrew': sum(page['hebrew_letter_count'] > 0 for page in pages)})

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

json_dir = 'Files/silver/pdf_pages'
json_path = f'{json_dir}/{doc_id}.json'
json_payload = dict(schema_version=1, document_id=doc_id, source_path=SOURCE_PATH,
                    extractor=EXTRACTOR, page_count=len(pages), pages=pages)
json_bytes = (json.dumps(json_payload, ensure_ascii=False, indent=2) + '\n').encode('utf-8')

bronze_table = 'bronze.pdf_documents'
silver_table = 'silver.pdf_pages'
bronze_schema = 'document_id STRING, source_path STRING, byte_size BIGINT, registered_at TIMESTAMP'
silver_schema = ('document_id STRING, page_number INT, text STRING, extraction_status STRING, '
                 'alphanumeric_count INT, hebrew_letter_count INT, suspect_character_count INT, extractor STRING')

for schema in ('bronze', 'silver'):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}')
for table_name, ddl in ((bronze_table, bronze_schema), (silver_table, silver_schema)):
    spark.sql(f'CREATE TABLE IF NOT EXISTS {table_name} ({ddl}) USING DELTA')

registered = (spark.table(bronze_table)
              .where((F.col('source_path') == SOURCE_PATH) | (F.col('document_id') == doc_id))
              .select('source_path', 'document_id').take(2))
if len(registered) > 1 or any(r.source_path != SOURCE_PATH or r.document_id != doc_id for r in registered):
    raise RuntimeError('Input conflicts with the immutable Bronze path or an already registered PDF. No page rows were changed.')

bronze = spark.createDataFrame([(doc_id, SOURCE_PATH, byte_size, datetime.now(timezone.utc))], bronze_schema)
silver = spark.createDataFrame([dict(document_id=doc_id, extractor=EXTRACTOR, **page) for page in pages], silver_schema)
(DeltaTable.forName(spark, bronze_table).alias('target')
 .merge(bronze.alias('incoming'), 'target.document_id = incoming.document_id')
 .whenNotMatchedInsertAll().execute())
(DeltaTable.forName(spark, silver_table).alias('target')
 .merge(silver.alias('incoming'), 'target.document_id = incoming.document_id AND target.page_number = incoming.page_number')
 .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

if not notebookutils.fs.mkdirs(json_dir):
    raise OSError('Could not prepare the Silver JSON folder.')
out = LAKEHOUSE / json_path
if not out.is_file() or out.read_bytes() != json_bytes:
    if not notebookutils.fs.put(json_path, json_bytes.decode('utf-8'), True):
        raise OSError('Silver Delta is written but the Silver JSON write failed; rerun the notebook.')

bronze_count = spark.table(bronze_table).where(F.col('document_id') == doc_id).count()
stored_pages = spark.table(silver_table).where(F.col('document_id') == doc_id).select(*silver.columns)
persisted_count = stored_pages.count()
if bronze_count != 1 or persisted_count != len(pages) or silver.exceptAll(stored_pages).limit(1).count():
    raise RuntimeError('Bronze or Silver Delta validation failed for this document.')
if out.read_bytes() != json_bytes:
    raise RuntimeError('Silver JSON read-back did not match the extracted pages.')
print({'bronze_documents': bronze_count, 'silver_delta_pages': persisted_count,
       'silver_json_pages': len(json_payload['pages'])})